<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step03aCropCowsTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 03a · Cut out the cows: 2-minute test

Takes the tracks from **Step 02b** (`Step02bFull/Tracks.csv`) and, for only the tracks in the **first 2 minutes**, 8 crops per track (~2 min). Use it to check the settings before Step 03b.

1. drops duplicate boxes (a weaker box inside a stronger one in the same frame),
2. keeps only tracks that are long and confident enough,
3. picks frames spread over each track, so the crops show different angles,
4. masks each cow with SAM 2.1 (pen, calf and other cows turn grey) and saves the crop.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step03aTest/`, **overwriting** each run:

- `KeptTracks.csv`: every track, kept or not and why; `RelativeSize` / `MaybeCalf` flag small boxes (calf, or the cow lying down)
- `Crops.csv` + `Crops/Track<id>/`: the masked crops
- `Sheets/Overview.jpg` (one crop per kept track) and `Sheets/Track<id>.jpg` (all crops of a track): check these by eye

Use a GPU runtime (**T4**) for SAM. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO/SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.CropCows import CropCows
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"            # step 01
TracksPath = VideoOutputDir / "Step02bFull" / "Tracks.csv"  # step 02b
OutputDir = VideoOutputDir / "Step03aTest"

MinFrames = 25         # drop tracks shorter than this (25 frames = 5 s at 5 fps)
MinConfidence = 0.4    # drop tracks whose average confidence is below this
CropsPerTrack = 8      # crops per kept track, spread over its time span
MaxSeconds = 120       # only the first 2 minutes
UseMasks = True        # False = plain box crops, no SAM (faster, background kept)

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"
assert TracksPath.exists(), f"Run step 02b first (or move its files into Step02bFull/): {TracksPath}"

In [ ]:
# 3. Run: crop + mask, then show which tracks were kept and the overview sheet.
import pandas as pd
from IPython.display import Image, display

Stats = CropCows(ClipPath, TracksPath, OutputDir, MinFrames=MinFrames, MinConfidence=MinConfidence,
                 CropsPerTrack=CropsPerTrack, MaxSeconds=MaxSeconds, UseMasks=UseMasks)

display(pd.read_csv(OutputDir / "KeptTracks.csv"))
display(Image(str(OutputDir / "Sheets" / "Overview.jpg")))
print("Per-track sheets:", OutputDir / "Sheets")